In [0]:
from pyspark.sql import functions as F

bronze_df = spark.read.table("bronze_orders")

print("Bronze record count:", bronze_df.count())

display(bronze_df)

In [0]:
null_checks = bronze_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in bronze_df.columns
])

display(null_checks)

In [0]:
duplicate_orders = (
    bronze_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_orders)

print("Duplicate order IDs:", duplicate_orders.count())

In [0]:
invalid_orders = bronze_df.filter(
    (F.col("quantity") <= 0) |
    (F.col("price") < 0) |
    F.col("order_id").isNull() |
    F.col("customer_id").isNull() |
    F.col("product").isNull() |
    F.col("order_date").isNull()
)

display(invalid_orders)

print("Invalid order count:", invalid_orders.count())

In [0]:
quality_summary = spark.createDataFrame([
    ("total_records", bronze_df.count()),
    ("duplicate_order_ids", duplicate_orders.count()),
    ("invalid_orders", invalid_orders.count())
], ["check_name", "check_value"])

display(quality_summary)